# Week 6, Day 4: Scope: local and global names

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w006d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def f(x):
    x = x * 2
    return x

x = 5
print(f(x))
print(x)

In [ ]:
def average(a, b):
    total = a + b
    result = total / 2
    return result

mean = average(4, 7)
print(mean)

In [ ]:
def average(a, b):
    total = a + b
    return total / 2

average(4, 7)
print(total)

In [ ]:
def counter():
    count = 0
    count += 1
    return count

print(counter(), counter(), counter())

In [ ]:
def area(width, height):
    result = width * height
    return result

def perimeter(width, height):
    result = 2 * (width + height)
    return result

result = "the main program's result"
print(area(3, 4), perimeter(3, 4))
print(result)

> **Key idea.** Names created inside a function are **local**: they belong to one call and vanish when it returns. A function can't accidentally change your variables by using the same names.

In [ ]:
VAT_RATE = 0.075

def with_vat(price):
    return price * (1 + VAT_RATE)

print(with_vat(1000))

> **Tip.** Global constants are fine. Global **variables** that functions depend on and change are not: they make a function's behaviour depend on things you can't see in its call. Pass values in as parameters instead.

In [ ]:
score = 10

def reset():
    score = 0
    print("Inside:", score)

reset()
print("Outside:", score)

In [ ]:
count = 0

def add_one():
    count = count + 1

add_one()

In [ ]:
def add_one(count):
    return count + 1

count = 0
count = add_one(count)
count = add_one(count)
print(count)

> **Note.** You'll meet `global` in other people's code, so here's how it reads: `global count` as the first line of a function means "`count` in this function is the global one". It works, but it's a sign the design could be better. Later in the course you'll learn better tools for sharing state, such as classes.

In [ ]:
sum = 0
for n in [1, 2, 3]:
    sum += n
print(sum([4, 5]))

Run the next cell to undo the damage, so the rest of this notebook works:

In [ ]:
del sum

### Exercise: Which x?

Work out what this program prints, line by line.

```python
x = "global"

def f():
    x = "local"
    return x

def g():
    return x

print(f())
print(g())
print(x)
```

In [ ]:
line_1 = "___"
line_2 = "___"
line_3 = "___"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_lines():\n    \"each line\"\n    assert line_1 == \"local\", \"f makes its own local x.\"\n    assert line_2 == \"global\", \"g doesn't assign x, so it reads the global one.\"\n    assert line_3 == \"global\", \"f's local x never touched the global.\"")

### Exercise: Fix the counter

This crashes with `UnboundLocalError`. Rewrite `add_point` so it takes the current score as a parameter and **returns** the new score, and change the calls to use it. The program should print `Score: 3`. Don't use `global`.

In [ ]:
score = 0

def add_point():
    score = score + 1

add_point()
add_point()
add_point()
print("Score:", score)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"Score: 3\"\n    expect_output(\"Score: 3\")\ndef test_design():\n    \"no global, and add_point returns\"\n    assert \"global\" not in source, \"Pass the score in and return it.\"\n    assert add_point(5) == 6, \"add_point(5) should return 6.\"")

### Exercise: No hidden inputs

`total_cost` secretly depends on the global variables `price` and `qty`. Rewrite it to take them as parameters, so that it works for any values, and keep the output the same.

In [ ]:
price = 2500
qty = 3

def total_cost():
    return price * qty

print(total_cost())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"same output\"\n    expect_output(\"7500\")\ndef test_params():\n    \"takes its inputs as parameters\"\n    assert total_cost(100, 4) == 400, \"total_cost(100, 4) should return 400.\"")

### Exercise: A constant

Define a constant `SPEED_LIMIT = 50` at the top of the program, and a function `fine(speed)` that returns 0 if the speed is at or below the limit, and otherwise 2000 for every whole 10 km/h (or part of 10) over the limit. So 51 to 60 is 2000, 61 to 70 is 4000. The checks will change the constant, so use it, not 50.

In [ ]:
SPEED_LIMIT = 50

def fine(speed):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several speeds\"\n    for s, f in [(40, 0), (50, 0), (51, 2000), (60, 2000), (61, 4000), (95, 10000)]:\n        assert fine(s) == f, f\"fine({s}) should be {f}.\"\ndef test_constant():\n    \"uses the constant\"\n    g = fine.__globals__\n    old = g[\"SPEED_LIMIT\"]\n    g[\"SPEED_LIMIT\"] = 30\n    try:\n        got = fine(45)\n    finally:\n        g[\"SPEED_LIMIT\"] = old\n    assert got == 4000, \"With SPEED_LIMIT = 30, fine(45) should be 4000. Use the constant, not 50.\"")

### Exercise: Same names, different jobs

Define `mean(a, b, c)` and `spread(a, b, c)`, each using a local variable called `result`. `mean` returns the average; `spread` returns the largest minus the smallest. Then print `Mean: 5.0` and `Spread: 6` for the numbers 2, 5 and 8, and finally print `result` from the main program, which you set to `"untouched"` at the start.

In [ ]:
result = "untouched"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"three lines\"\n    expect_output(\"Mean: 5.0\\nSpread: 6\\nuntouched\")\ndef test_functions():\n    \"both functions work\"\n    assert mean(1, 2, 3) == 2 and spread(1, 10, 4) == 9\n    assert source.count(\"result =\") >= 3, \"Use a local called result in each function.\"")

### Bug hunt: A local that escaped

This crashes with a `NameError`. Fix it so the program prints `Discounted: 900.0`, by using the function's return value.

In [ ]:
def discount(price):
    new_price = price * 0.9
    return new_price

discount(1000)
print("Discounted:", new_price)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"prints the discounted price\"\n    expect_output(\"Discounted: 900.0\")")

### Bug hunt: max is broken

This should print the total and the largest value, but it crashes. Fix the variable name without changing what's printed.

In [ ]:
values = "4 9 2 7"
max = 0
for v in values.split():
    max += int(v)
print("Total:", max)
print("Largest:", max(4, 9, 2, 7))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"total and largest\"\n    expect_output(\"Total: 22\\nLargest: 9\")")

### Bug hunt: The forgetful counter

This should count the vowels in two words and print `Vowels: 5`, but it prints `Vowels: 0`. The function counts correctly. Fix the program without using `global`.

In [ ]:
total = 0

def count_vowels(word):
    total = 0
    for ch in word:
        if ch in "aeiou":
            total += 1

count_vowels("banana")
count_vowels("kiwi")
print("Vowels:", total)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"5 vowels\"\n    expect_output(\"Vowels: 5\")\ndef test_no_global():\n    \"no global\"\n    assert \"global\" not in source\n    assert count_vowels(\"aeiou\") == 5, \"count_vowels should return its count.\"")

### Puzzle: In through parameters, out through return

Arrange the lines into a function that applies interest to a balance and returns it, used twice by the main program. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Trace the frames

What does this print? Write each line.

```python
n = 1

def a(n):
    n = n + 10
    return b(n) + n

def b(m):
    return m * n

print(a(2))
print(n)
```

In [ ]:
line_1 = ___
line_2 = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_lines():\n    \"both lines\"\n    assert line_1 == 24, \"In a, n is 12. b(12) uses the global n, which is 1, so returns 12. Then 12 + 12.\"\n    assert line_2 == 1, \"The global n never changed.\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a bank account with no globals

Write three functions, none of which uses `global` or prints anything:

- `deposit(balance, amount)` returns the new balance;
- `withdraw(balance, amount)` returns the new balance, or the unchanged balance if there isn't enough money;
- `apply_interest(balance, rate_percent)` returns the balance increased by that percentage, rounded to 2 places.

Then read commands with the prompt `> ` until `end`. A command is `d 500` (deposit), `w 200` (withdraw) or `i 5` (interest at 5%). After each command print `Balance: ` and the balance with 2 decimal places. The balance starts at 0.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_functions():\n    \"the three functions\"\n    assert deposit(100, 50) == 150\n    assert withdraw(100, 30) == 70 and withdraw(100, 300) == 100\n    assert apply_interest(1000, 5) == 1050.0 and apply_interest(333.33, 10) == 366.66\ndef test_session():\n    \"a session\"\n    expect_output(\"> d 1000\\nBalance: 1000.00\\n> w 300\\nBalance: 700.00\\n> w 900\\nBalance: 700.00\\n> i 10\\nBalance: 770.00\\n> end\",\n                  inputs=[\"d 1000\", \"w 300\", \"w 900\", \"i 10\", \"end\"], echo=True)\ndef test_style():\n    \"no globals or prints in the functions\"\n    assert \"global\" not in source\n    for name in [\"def deposit\", \"def withdraw\", \"def apply_interest\"]:\n        start = source.find(name)\n        nxt = source.find(\"\\ndef \", start + 1)\n        body = source[start: nxt if nxt != -1 else len(source)]\n        body = body.split(\"\\n\\n\")[0]\n        assert \"print\" not in body, name + \" should return, not print.\"")

## Warm-up

Two different `x`s are at work here: one in the function and one outside. Yesterday you saw that changing the function's `x` didn't change the other. Today you'll learn the rules behind that.

## Local names live in the frame

Every time a function is called, Python creates a fresh **frame** for that call. Every name the function creates, its parameters and any variable it assigns, lives in that frame. These are **local** names. When the function returns, the frame is thrown away, and the local names with it.

Step through it. During the call, the frame for `average` holds `a`, `b`, `total` and `result`. When `return` runs, the frame disappears. Only the returned **value** survives, and the main program stores it in `mean`.

So a local name can't be seen from outside:

`NameError: name 'total' is not defined`. `total` existed only during the call.

### Every call is fresh

Local names don't survive from one call to the next. Each call starts with a brand new frame:

It prints `1 1 1`, not `1 2 3`. Each call creates its own `count`, sets it to 0, adds 1, and throws it away.

### Same names, no clash

Because each function has its own frame, different functions can use the same local names without interfering with each other or with the main program:

Three different `result`s, in three different places, and none of them touches the others. This is the whole point of scope: you can write or use a function without knowing or caring which names it uses inside.

## Reading global names

Names created at the top level of a program, outside any function, are **global**. A function can **read** them:

When `with_vat` uses `VAT_RATE`, Python looks in the function's frame first, doesn't find it there, and then looks in the global names, where it is.

Reading a global **constant** like this is perfectly normal. By convention, constants are written in `UPPER_CASE` to say "this never changes".

## Assigning makes a name local

Here's the rule that surprises people. If a function **assigns** to a name anywhere in its body, that name is local **everywhere** in that function, even on lines before the assignment.

`score = 0` inside the function creates a **new** local `score`. The global one is untouched. The local one **shadows** the global: inside the function, it hides it.

Now the error this causes:

`UnboundLocalError: cannot access local variable 'count' where it is not associated with a value`. Because the function assigns to `count`, `count` is local throughout the function. So `count + 1` looks for the **local** `count`, which doesn't have a value yet.

### The fix: parameters in, return out

You could tell Python to use the global with a `global count` line. Don't. It makes the function depend on, and secretly change, a variable outside it. Instead, pass the value in and return the new value:

Now everything the function needs comes in through its parameters, and everything it produces goes out through `return`. You can understand it, test it and reuse it just by reading its `def` line and its body.

## How Python finds a name

When Python meets a name, it looks for it in this order, and uses the first one it finds:

1. **Local**: the current function's frame.
2. **Enclosing**: the frames of any functions this one is defined inside. (You'll meet functions inside functions in Week 7.)
3. **Global**: the top level of the program.
4. **Built-in**: the names Python provides, like `print`, `len` and `max`.

This is called the **LEGB rule**. If no level has the name, you get a `NameError`.

The last level explains a trap you met in Week 1. Built-in names can be shadowed like any other:

After `sum = 0`, the global `sum` is found before the built-in one, so `sum([4, 5])` tries to call a number. Avoid `sum`, `max`, `min`, `list`, `str`, `input` and `print` as variable names.

## Practice

## Quiz

## Daily challenge

## Before you go

Draw the frames for the trace exercise on paper, the way X-ray shows them, at the moment `b` is running. Then state the LEGB rule from memory, and explain why "parameters in, return out" makes functions easier to trust. Tomorrow: docstrings, how to describe a function so that people, and Python's own help system, know exactly how to use it.